# Stage 1 Attention Mechanism Analysis

This notebook is self-contained for Google Colab or another machine. It does not import repository files. Upload only `model.pt` to analyze the default Stage 1 model. If the model architecture changed, also upload the matching `config.json`.

## 1. Dependencies

Colab usually already has PyTorch installed. If your runtime does not have PyTorch, uncomment the install line.

In [ ]:
# Uncomment only if torch is missing in your runtime.
# %pip install torch

import csv
import json
import math
from dataclasses import asdict, dataclass, fields
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn

try:
    import pandas as pd
except ImportError:
    pd = None

torch.set_grad_enabled(False)
print(f'torch={torch.__version__}, cuda_available={torch.cuda.is_available()}')

## 2. Upload Or Point To The Model

For Colab, upload `model.pt` here. Upload `config.json` only if the model does not use the default Stage 1 config. If your checkpoint filename is different, change `MODEL_PATH` below.

In [ ]:
# Colab upload path. This block is safe to skip outside Colab.
try:
    from google.colab import files
    print('Colab detected. Upload model.pt, and optionally config.json.')
    uploaded = files.upload()
except Exception:
    uploaded = {}

# Defaults for Colab uploads. For local use, replace these paths manually.
MODEL_PATH = Path('model.pt')
CONFIG_PATH = Path('config.json') if Path('config.json').exists() else None
OUTPUT_DIR = Path('numerical_analysis_notebook')
LENGTHS = [10, 100, 500, 900, 1000, 1100]
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('MODEL_PATH =', MODEL_PATH)
print('CONFIG_PATH =', CONFIG_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('DEVICE =', DEVICE)

if not MODEL_PATH.exists():
    checkpoint_candidates = sorted(Path('.').glob('*.pt')) + sorted(Path('.').glob('*.pth'))
    raise FileNotFoundError(
        f'MODEL_PATH does not exist: {MODEL_PATH}. '
        f'Available checkpoint candidates: {[str(path) for path in checkpoint_candidates]}. '
        'Upload model.pt or edit MODEL_PATH.'
    )

## 3. Self-Contained Model, Data, And Analysis Code

This cell intentionally duplicates the minimal project code so the notebook can run without the repository source tree.

In [ ]:
@dataclass(frozen=True)
class TaskConfig:
    vocab_size: int = 16
    target_token: int = 1
    eval_lengths: tuple[int, ...] = (10, 20, 50, 100, 200, 500, 700, 800, 850, 900, 950, 1000, 1100)
    positive_fraction: float = 0.5

    def __post_init__(self):
        object.__setattr__(self, 'eval_lengths', tuple(self.eval_lengths))

    def to_dict(self):
        data = asdict(self)
        data['eval_lengths'] = list(self.eval_lengths)
        return data


@dataclass(frozen=True)
class Stage1Config:
    seed: int = 1234
    device: str = 'auto'
    train_length: int = 10
    train_examples: int = 50000
    val_examples: int = 10000
    test_examples: int = 10000
    diagnostic_examples: int = 2000
    batch_size: int = 512
    eval_batch_size: int = 32
    epochs: int = 10
    learning_rate: float = 1e-3
    weight_decay: float = 0.0
    d_model: int = 64
    num_heads: int = 1
    num_layers: int = 1
    dim_feedforward: int = 128
    dropout: float = 0.0
    output_dir: str = 'runs/stage1_transformer_maxpool'

    def to_dict(self):
        return asdict(self)


def dataclass_values_from_metadata(config_class, values):
    valid_names = {field.name for field in fields(config_class)}
    return {key: value for key, value in values.items() if key in valid_names}


class AttentionExportEncoderLayer(nn.Module):
    def __init__(self, *, d_model, num_heads, dim_feedforward, dropout):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.linear2 = nn.Linear(dim_feedforward, d_model)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.activation = nn.ReLU()

    def forward(self, hidden, *, return_attention):
        attention_output, attention_weights = self.self_attn(
            hidden,
            hidden,
            hidden,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        hidden = self.norm1(hidden + self.dropout1(attention_output))
        feedforward_output = self.linear2(self.dropout(self.activation(self.linear1(hidden))))
        hidden = self.norm2(hidden + self.dropout2(feedforward_output))
        return hidden, attention_weights


class TransformerTokenPresenceClassifier(nn.Module):
    def __init__(self, *, vocab_size, d_model, num_heads, num_layers, dim_feedforward, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([
            AttentionExportEncoderLayer(
                d_model=d_model,
                num_heads=num_heads,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
            )
            for _ in range(num_layers)
        ])
        self.classifier = nn.Linear(d_model, 1)

    def encode(self, tokens, *, return_attention):
        hidden = self.embedding(tokens)
        attention_weights = []
        for layer in self.layers:
            hidden, weights = layer(hidden, return_attention=return_attention)
            if weights is not None:
                attention_weights.append(weights)
        return hidden, attention_weights

    def forward(self, tokens):
        encoded, _ = self.encode(tokens, return_attention=False)
        pooled = encoded.max(dim=1).values
        return self.classifier(pooled).squeeze(-1)


def _sample_non_target_tokens(shape, *, vocab_size, target_token, generator):
    tokens = torch.randint(0, vocab_size - 1, shape, generator=generator)
    return tokens + (tokens >= target_token).long()


def make_negative_dataset(*, num_examples, length, task, generator):
    inputs = _sample_non_target_tokens(
        (num_examples, length),
        vocab_size=task.vocab_size,
        target_token=task.target_token,
        generator=generator,
    )
    labels = torch.zeros(num_examples, dtype=torch.float32)
    return inputs, labels


def _sample_target_positions(*, num_examples, length, target_count, region, generator):
    if region == 'random':
        candidates = torch.arange(length)
    elif region == 'begin':
        candidates = torch.arange(0, max(1, length // 3))
    elif region == 'middle':
        start = length // 3
        end = max(start + 1, (2 * length) // 3)
        candidates = torch.arange(start, end)
    elif region == 'end':
        candidates = torch.arange((2 * length) // 3, length)
    else:
        raise ValueError(f'unknown target region: {region}')
    rows = []
    for _ in range(num_examples):
        permutation = torch.randperm(len(candidates), generator=generator)
        rows.append(candidates[permutation[:target_count]])
    return torch.stack(rows, dim=0)


def make_positive_dataset(*, num_examples, length, task, generator, target_count=1, target_region='random'):
    inputs = _sample_non_target_tokens(
        (num_examples, length),
        vocab_size=task.vocab_size,
        target_token=task.target_token,
        generator=generator,
    )
    positions = _sample_target_positions(
        num_examples=num_examples,
        length=length,
        target_count=target_count,
        region=target_region,
        generator=generator,
    )
    inputs[torch.arange(num_examples).unsqueeze(1), positions] = task.target_token
    labels = torch.ones(num_examples, dtype=torch.float32)
    return inputs, labels


def make_controlled_sequence(*, length, label_type, task, seed):
    generator = torch.Generator().manual_seed(seed)
    if label_type == 'positive':
        inputs, _ = make_positive_dataset(
            num_examples=1,
            length=length,
            task=task,
            generator=generator,
            target_count=1,
            target_region='middle',
        )
        target_index = int(torch.nonzero(inputs[0].eq(task.target_token), as_tuple=False)[0].item())
        return inputs[0], target_index
    if label_type == 'negative':
        inputs, _ = make_negative_dataset(num_examples=1, length=length, task=task, generator=generator)
        return inputs[0], None
    raise ValueError(f'unknown label type: {label_type}')

In [ ]:
def split_qkv(layer):
    weight = layer.self_attn.in_proj_weight.detach()
    bias = layer.self_attn.in_proj_bias.detach()
    w_q, w_k, w_v = weight.chunk(3, dim=0)
    b_q, b_k, b_v = bias.chunk(3, dim=0)
    return {'w_q': w_q, 'w_k': w_k, 'w_v': w_v, 'b_q': b_q, 'b_k': b_k, 'b_v': b_v}


def write_csv(path, rows):
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return
    fieldnames = list(rows[0])
    for row in rows[1:]:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with path.open('w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


def parameter_stat_rows(model):
    rows = []
    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue
        values = parameter.detach().float().cpu()
        rows.append({
            'name': name,
            'shape': list(parameter.shape),
            'count': parameter.numel(),
            'l2_norm': float(torch.linalg.vector_norm(values).item()),
            'mean': float(values.mean().item()),
            'std': float(values.std(unbiased=False).item()),
            'min': float(values.min().item()),
            'max': float(values.max().item()),
        })
    return rows


def safe_cosine(a, b):
    denominator = torch.linalg.vector_norm(a) * torch.linalg.vector_norm(b)
    if denominator.item() == 0.0:
        return float('nan')
    return float(torch.dot(a, b).div(denominator).item())


def token_qkv_geometry_rows(model, task):
    layer = model.layers[0]
    qkv = split_qkv(layer)
    embeddings = model.embedding.weight.detach()
    q = F.linear(embeddings, qkv['w_q'], qkv['b_q'])
    k = F.linear(embeddings, qkv['w_k'], qkv['b_k'])
    v = F.linear(embeddings, qkv['w_v'], qkv['b_v'])
    target = task.target_token
    d_head = layer.self_attn.head_dim
    rows = []
    for token_id in range(task.vocab_size):
        rows.append({
            'token_id': token_id,
            'is_target': token_id == target,
            'q_norm': float(torch.linalg.vector_norm(q[token_id]).item()),
            'k_norm': float(torch.linalg.vector_norm(k[token_id]).item()),
            'v_norm': float(torch.linalg.vector_norm(v[token_id]).item()),
            'q_cosine_to_target': safe_cosine(q[token_id], q[target]),
            'k_cosine_to_target': safe_cosine(k[token_id], k[target]),
            'v_cosine_to_target': safe_cosine(v[token_id], v[target]),
            'query_to_target_key_score': float(torch.dot(q[token_id], k[target]).div(math.sqrt(d_head)).item()),
            'target_query_to_key_score': float(torch.dot(q[target], k[token_id]).div(math.sqrt(d_head)).item()),
        })
    return rows


@torch.no_grad()
def manual_layer0_forward(model, tokens):
    if len(model.layers) != 1:
        raise ValueError('this notebook expects exactly one transformer layer')
    device = next(model.parameters()).device
    tokens = tokens.to(device)
    layer = model.layers[0]
    hidden = model.embedding(tokens.unsqueeze(0)).squeeze(0)
    qkv = split_qkv(layer)
    q_full = F.linear(hidden, qkv['w_q'], qkv['b_q'])
    k_full = F.linear(hidden, qkv['w_k'], qkv['b_k'])
    v_full = F.linear(hidden, qkv['w_v'], qkv['b_v'])
    num_heads = layer.self_attn.num_heads
    d_head = layer.self_attn.head_dim
    length = hidden.shape[0]
    q = q_full.view(length, num_heads, d_head).transpose(0, 1)
    k = k_full.view(length, num_heads, d_head).transpose(0, 1)
    v = v_full.view(length, num_heads, d_head).transpose(0, 1)
    scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(d_head)
    attention = torch.softmax(scores, dim=-1)
    attention_pre_out = torch.matmul(attention, v).transpose(0, 1).reshape(length, -1)
    attention_out = layer.self_attn.out_proj(attention_pre_out)
    after_attention = layer.norm1(hidden + attention_out)
    feedforward = layer.linear2(layer.dropout(layer.activation(layer.linear1(after_attention))))
    encoded = layer.norm2(after_attention + feedforward)
    pooled, argmax_positions = encoded.max(dim=0)
    logit = model.classifier(pooled).squeeze()
    return {
        'embedding': hidden,
        'q': q,
        'k': k,
        'v': v,
        'scores': scores,
        'attention': attention,
        'attention_pre_out': attention_pre_out,
        'attention_out': attention_out,
        'after_attention': after_attention,
        'feedforward': feedforward,
        'encoded': encoded,
        'pooled': pooled,
        'argmax_positions': argmax_positions,
        'logit': logit,
    }


def entropy(attention):
    safe_attention = attention.clamp_min(1e-12)
    return -(safe_attention * safe_attention.log()).sum(dim=-1)

In [ ]:
def attention_length_rows(model, task, lengths):
    rows = []
    for length_index, length in enumerate(lengths):
        tokens, target_index = make_controlled_sequence(length=length, label_type='positive', task=task, seed=50_000 + length_index)
        tensors = manual_layer0_forward(model, tokens)
        scores = tensors['scores'][0].detach().cpu()
        attention = tensors['attention'][0].detach().cpu()
        target_scores = scores[:, target_index]
        non_target_mask = torch.ones(length, dtype=torch.bool)
        non_target_mask[target_index] = False
        non_target_scores = scores[:, non_target_mask]
        denominator = scores.exp().sum(dim=-1)
        target_exp = target_scores.exp()
        non_target_exp_sum = non_target_scores.exp().sum(dim=-1)
        target_mass = attention[:, target_index]
        query_entropy = entropy(attention)
        max_query = int(target_mass.argmax().item())
        for query_name, query_index in (('mean_over_queries', None), ('target_query', target_index), ('last_query', length - 1), ('max_target_attention_query', max_query)):
            if query_index is None:
                rows.append({
                    'length': length,
                    'query': query_name,
                    'target_index': target_index,
                    'logit': float(tensors['logit'].detach().cpu().item()),
                    'target_score_mean': float(target_scores.mean().item()),
                    'target_score_max': float(target_scores.max().item()),
                    'non_target_score_mean': float(non_target_scores.mean().item()),
                    'non_target_score_max': float(non_target_scores.max().item()),
                    'target_minus_mean_non_target': float((target_scores - non_target_scores.mean(dim=-1)).mean().item()),
                    'target_minus_max_non_target': float((target_scores - non_target_scores.max(dim=-1).values).mean().item()),
                    'target_exp_mean': float(target_exp.mean().item()),
                    'non_target_exp_sum_mean': float(non_target_exp_sum.mean().item()),
                    'softmax_denominator_mean': float(denominator.mean().item()),
                    'target_attention_mean': float(target_mass.mean().item()),
                    'target_attention_max': float(target_mass.max().item()),
                    'attention_entropy_mean': float(query_entropy.mean().item()),
                    'attention_entropy_max': float(query_entropy.max().item()),
                })
            else:
                query_scores = scores[query_index]
                target_score = query_scores[target_index]
                non_target_query_scores = query_scores[non_target_mask]
                rows.append({
                    'length': length,
                    'query': query_name,
                    'target_index': target_index,
                    'query_index': query_index,
                    'logit': float(tensors['logit'].detach().cpu().item()),
                    'target_score': float(target_score.item()),
                    'non_target_score_mean': float(non_target_query_scores.mean().item()),
                    'non_target_score_max': float(non_target_query_scores.max().item()),
                    'target_minus_mean_non_target': float((target_score - non_target_query_scores.mean()).item()),
                    'target_minus_max_non_target': float((target_score - non_target_query_scores.max()).item()),
                    'target_exp': float(target_score.exp().item()),
                    'non_target_exp_sum': float(non_target_query_scores.exp().sum().item()),
                    'softmax_denominator': float(query_scores.exp().sum().item()),
                    'target_attention': float(attention[query_index, target_index].item()),
                    'attention_entropy': float(query_entropy[query_index].item()),
                })
    return rows


def hidden_evidence_rows(model, task, lengths):
    rows = []
    classifier_weight = model.classifier.weight.detach().squeeze(0).cpu()
    tensor_names = ['embedding', 'attention_pre_out', 'attention_out', 'after_attention', 'feedforward', 'encoded']
    for length_index, length in enumerate(lengths):
        tokens, target_index = make_controlled_sequence(length=length, label_type='positive', task=task, seed=50_000 + length_index)
        tensors = manual_layer0_forward(model, tokens)
        non_target_mask = torch.ones(length, dtype=torch.bool)
        non_target_mask[target_index] = False
        for tensor_name in tensor_names:
            values = tensors[tensor_name].detach().cpu()
            target_vector = values[target_index]
            non_target_values = values[non_target_mask]
            evidence = values @ classifier_weight
            rows.append({
                'length': length,
                'tensor': tensor_name,
                'target_index': target_index,
                'logit': float(tensors['logit'].detach().cpu().item()),
                'target_norm': float(torch.linalg.vector_norm(target_vector).item()),
                'non_target_norm_mean': float(torch.linalg.vector_norm(non_target_values, dim=1).mean().item()),
                'non_target_norm_max': float(torch.linalg.vector_norm(non_target_values, dim=1).max().item()),
                'target_classifier_evidence': float(evidence[target_index].item()),
                'non_target_classifier_evidence_mean': float(evidence[non_target_mask].mean().item()),
                'non_target_classifier_evidence_max': float(evidence[non_target_mask].max().item()),
            })
    return rows


def maxpool_and_logit_rows(model, task, lengths):
    maxpool_rows = []
    logit_rows = []
    classifier_weight = model.classifier.weight.detach().squeeze(0).cpu()
    classifier_bias = float(model.classifier.bias.detach().squeeze().cpu().item())
    for length_index, length in enumerate(lengths):
        tokens, target_index = make_controlled_sequence(length=length, label_type='positive', task=task, seed=50_000 + length_index)
        tensors = manual_layer0_forward(model, tokens)
        pooled = tensors['pooled'].detach().cpu()
        argmax_positions = tensors['argmax_positions'].detach().cpu()
        contributions = classifier_weight * pooled
        target_dim_mask = argmax_positions.eq(target_index)
        non_target_dim_mask = ~target_dim_mask
        positive_contributions = contributions.clamp_min(0)
        negative_contributions = contributions.clamp_max(0)
        maxpool_rows.append({
            'length': length,
            'target_index': target_index,
            'logit': float(tensors['logit'].detach().cpu().item()),
            'target_sourced_dim_count': int(target_dim_mask.sum().item()),
            'non_target_sourced_dim_count': int(non_target_dim_mask.sum().item()),
            'target_sourced_dim_fraction': float(target_dim_mask.float().mean().item()),
            'target_sourced_contribution_sum': float(contributions[target_dim_mask].sum().item()),
            'non_target_sourced_contribution_sum': float(contributions[non_target_dim_mask].sum().item()),
            'target_sourced_positive_contribution_sum': float(positive_contributions[target_dim_mask].sum().item()),
            'non_target_sourced_positive_contribution_sum': float(positive_contributions[non_target_dim_mask].sum().item()),
            'target_sourced_negative_contribution_sum': float(negative_contributions[target_dim_mask].sum().item()),
            'non_target_sourced_negative_contribution_sum': float(negative_contributions[non_target_dim_mask].sum().item()),
        })
        logit_rows.append({
            'length': length,
            'target_index': target_index,
            'logit': float(tensors['logit'].detach().cpu().item()),
            'classifier_bias': classifier_bias,
            'contribution_sum': float(contributions.sum().item()),
            'positive_contribution_sum': float(positive_contributions.sum().item()),
            'negative_contribution_sum': float(negative_contributions.sum().item()),
        })
    return maxpool_rows, logit_rows

## 4. Load The Checkpoint

If `config.json` is not uploaded, the notebook uses the default Stage 1 config.

In [ ]:
def load_stage1_model_from_files(model_path, config_path=None, device=torch.device('cpu')):
    if config_path is not None and Path(config_path).exists():
        metadata = json.loads(Path(config_path).read_text(encoding='utf-8'))
        task = TaskConfig(**dataclass_values_from_metadata(TaskConfig, metadata.get('task', {})))
        config = Stage1Config(**dataclass_values_from_metadata(Stage1Config, metadata.get('stage1', {})))
    else:
        task = TaskConfig()
        config = Stage1Config()
        metadata = {'task': task.to_dict(), 'stage1': config.to_dict(), 'source': 'defaults'}
    model = TransformerTokenPresenceClassifier(
        vocab_size=task.vocab_size,
        d_model=config.d_model,
        num_heads=config.num_heads,
        num_layers=config.num_layers,
        dim_feedforward=config.dim_feedforward,
        dropout=config.dropout,
    ).to(device)
    state_dict = torch.load(model_path, map_location=device)
    model.load_state_dict(state_dict)
    model.eval()
    return model, task, config, metadata


model, task, config, metadata = load_stage1_model_from_files(MODEL_PATH, CONFIG_PATH, DEVICE)
print(f'Loaded model from: {MODEL_PATH}')
print(f'Device: {DEVICE}')
print(f'Vocab size: {task.vocab_size}')
print(f'Target token: {task.target_token}')
print(f'd_model={config.d_model}, heads={config.num_heads}, layers={config.num_layers}')

## 5. Run Analysis

In [ ]:
@torch.no_grad()
def controlled_example_rows(model, task, lengths):
    rows = []
    for length_index, length in enumerate(lengths):
        for label_type, label in (('negative', 0), ('positive', 1)):
            seed = 50_000 + length_index if label_type == 'positive' else 70_000 + length_index
            tokens, target_index = make_controlled_sequence(length=length, label_type=label_type, task=task, seed=seed)
            tensors = manual_layer0_forward(model, tokens)
            logit = float(tensors['logit'].detach().cpu().item())
            pooled = tensors['pooled'].detach().cpu()
            rows.append({
                'length': length,
                'label_type': label_type,
                'label': label,
                'target_index': '' if target_index is None else target_index,
                'logit': logit,
                'probability': float(torch.sigmoid(torch.tensor(logit)).item()),
                'prediction': int(logit >= 0.0),
                'correct': int((logit >= 0.0) == bool(label)),
                'pooled_l2_norm': float(torch.linalg.vector_norm(pooled).item()),
                'pooled_abs_mean': float(pooled.abs().mean().item()),
                'pooled_dim_min': float(pooled.min().item()),
                'pooled_dim_max': float(pooled.max().item()),
            })
    return rows


def write_summary(path, lengths, controlled_rows, attention_rows, maxpool_rows):
    positive_rows = [row for row in controlled_rows if row['label_type'] == 'positive']
    mean_attention_rows = [row for row in attention_rows if row['query'] == 'mean_over_queries']
    lines = [
        '# Stage 1 Numerical Analysis Summary',
        '',
        f"Analyzed lengths: {', '.join(str(length) for length in lengths)}",
        '',
        '## Positive Logit By Length',
        '',
        '| Length | Logit | Probability | Prediction |',
        '|---:|---:|---:|---:|',
    ]
    for row in positive_rows:
        lines.append('| {length} | {logit:.4f} | {probability:.4f} | {prediction} |'.format(**row))
    lines.extend(['', '## Mean Target Attention By Length', '', '| Length | Target Attention Mean | Target Attention Max | Entropy Mean | Logit |', '|---:|---:|---:|---:|---:|'])
    for row in mean_attention_rows:
        lines.append('| {length} | {target_attention_mean:.4f} | {target_attention_max:.4f} | {attention_entropy_mean:.4f} | {logit:.4f} |'.format(**row))
    lines.extend(['', '## Max-Pool Target Source Fraction', '', '| Length | Target-Sourced Dim Fraction | Target Contribution Sum | Non-Target Contribution Sum |', '|---:|---:|---:|---:|'])
    for row in maxpool_rows:
        lines.append('| {length} | {target_sourced_dim_fraction:.4f} | {target_sourced_contribution_sum:.4f} | {non_target_sourced_contribution_sum:.4f} |'.format(**row))
    path.write_text('\n'.join(lines) + '\n', encoding='utf-8')


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
parameter_rows = parameter_stat_rows(model)
qkv_rows = token_qkv_geometry_rows(model, task)
controlled_rows = controlled_example_rows(model, task, LENGTHS)
attention_rows = attention_length_rows(model, task, LENGTHS)
evidence_rows = hidden_evidence_rows(model, task, LENGTHS)
maxpool_rows, logit_rows = maxpool_and_logit_rows(model, task, LENGTHS)

write_csv(OUTPUT_DIR / 'parameter_stats.csv', parameter_rows)
write_csv(OUTPUT_DIR / 'token_qkv_geometry.csv', qkv_rows)
write_csv(OUTPUT_DIR / 'controlled_examples.csv', controlled_rows)
write_csv(OUTPUT_DIR / 'attention_length_summary.csv', attention_rows)
write_csv(OUTPUT_DIR / 'hidden_evidence_summary.csv', evidence_rows)
write_csv(OUTPUT_DIR / 'maxpool_source_summary.csv', maxpool_rows)
write_csv(OUTPUT_DIR / 'logit_decomposition.csv', logit_rows)
(OUTPUT_DIR / 'metadata.json').write_text(json.dumps({'model_path': str(MODEL_PATH), 'config_path': '' if CONFIG_PATH is None else str(CONFIG_PATH), 'lengths': LENGTHS, 'task': task.to_dict(), 'stage1': config.to_dict(), 'source_metadata': metadata}, indent=2, sort_keys=True) + '\n', encoding='utf-8')
write_summary(OUTPUT_DIR / 'summary.md', LENGTHS, controlled_rows, attention_rows, maxpool_rows)
print(f'Saved analysis files to {OUTPUT_DIR}')

## 6. Inspect Key Tables

In [ ]:
def show_rows(rows, columns=None):
    if pd is not None:
        return pd.DataFrame(rows)[columns] if columns else pd.DataFrame(rows)
    return [{key: row[key] for key in columns} for row in rows] if columns else rows

show_rows([row for row in controlled_rows if row['label_type'] == 'positive'], ['length', 'logit', 'probability', 'prediction', 'correct', 'pooled_l2_norm'])

In [ ]:
show_rows([row for row in attention_rows if row['query'] == 'mean_over_queries'], ['length', 'logit', 'target_score_mean', 'non_target_exp_sum_mean', 'softmax_denominator_mean', 'target_attention_mean', 'target_attention_max', 'attention_entropy_mean'])

In [ ]:
show_rows(maxpool_rows, ['length', 'logit', 'target_sourced_dim_fraction', 'target_sourced_contribution_sum', 'non_target_sourced_contribution_sum'])

In [ ]:
show_rows(logit_rows, ['length', 'logit', 'positive_contribution_sum', 'negative_contribution_sum', 'classifier_bias'])

## 7. Save Figures For Markdown

These PNG files can be attached to `ANALYSIS.md` after the notebook run.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError as error:
    raise ImportError('matplotlib is required for figure export. Run `%pip install matplotlib` and retry this cell.') from error

FIGURE_DIR = OUTPUT_DIR / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def series(rows, key, *, label_type=None, query=None):
    filtered = rows
    if label_type is not None:
        filtered = [row for row in filtered if row.get('label_type') == label_type]
    if query is not None:
        filtered = [row for row in filtered if row.get('query') == query]
    filtered = sorted(filtered, key=lambda row: row['length'])
    return [row['length'] for row in filtered], [row[key] for row in filtered]

def save_line_figure(path, title, ylabel, lines, *, yline=None, log_x=True):
    fig, ax = plt.subplots(figsize=(7, 4.2), dpi=160)
    for label, xs, ys in lines:
        ax.plot(xs, ys, marker='o', linewidth=2, label=label)
    if yline is not None:
        ax.axhline(yline, color='black', linestyle='--', linewidth=1)
    if log_x:
        ax.set_xscale('log')
    ax.set_title(title)
    ax.set_xlabel('evaluation length')
    ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.25)
    if len(lines) > 1:
        ax.legend()
    fig.tight_layout()
    fig.savefig(path, bbox_inches='tight')
    plt.close(fig)

pos_x, pos_logit = series(controlled_rows, 'logit', label_type='positive')
neg_x, neg_logit = series(controlled_rows, 'logit', label_type='negative')
save_line_figure(FIGURE_DIR / 'controlled_logits_by_length.png', 'Controlled logits by length', 'logit', [('positive', pos_x, pos_logit), ('negative', neg_x, neg_logit)], yline=0.0)

pos_x, pos_prob = series(controlled_rows, 'probability', label_type='positive')
neg_x, neg_prob = series(controlled_rows, 'probability', label_type='negative')
save_line_figure(FIGURE_DIR / 'controlled_probabilities_by_length.png', 'Controlled probabilities by length', 'probability', [('positive', pos_x, pos_prob), ('negative', neg_x, neg_prob)], yline=0.5)

att_x, target_attention = series(attention_rows, 'target_attention_mean', query='mean_over_queries')
_, target_attention_max = series(attention_rows, 'target_attention_max', query='mean_over_queries')
save_line_figure(FIGURE_DIR / 'target_attention_by_length.png', 'Attention mass on target key', 'attention mass', [('mean', att_x, target_attention), ('max', att_x, target_attention_max)])

denom_x, denominator = series(attention_rows, 'softmax_denominator_mean', query='mean_over_queries')
_, non_target_exp = series(attention_rows, 'non_target_exp_sum_mean', query='mean_over_queries')
save_line_figure(FIGURE_DIR / 'softmax_denominator_by_length.png', 'Softmax denominator growth', 'mean exp-score sum', [('denominator', denom_x, denominator), ('non-target exp sum', denom_x, non_target_exp)])

pool_x, target_contrib = series(maxpool_rows, 'target_sourced_contribution_sum')
_, non_target_contrib = series(maxpool_rows, 'non_target_sourced_contribution_sum')
save_line_figure(FIGURE_DIR / 'maxpool_contributions_by_length.png', 'Classifier contribution by max-pool source', 'logit contribution', [('target-sourced dims', pool_x, target_contrib), ('non-target-sourced dims', pool_x, non_target_contrib)], yline=0.0)

logit_x, positive_contrib = series(logit_rows, 'positive_contribution_sum')
_, negative_contrib = series(logit_rows, 'negative_contribution_sum')
save_line_figure(FIGURE_DIR / 'logit_decomposition_by_length.png', 'Logit decomposition by length', 'logit contribution', [('positive contribution', logit_x, positive_contrib), ('negative contribution', logit_x, negative_contrib)], yline=0.0)

figure_paths = sorted(FIGURE_DIR.glob('*.png'))
print('Saved figures:')
for path in figure_paths:
    print('-', path)

In [ ]:
from IPython.display import Markdown, display
display(Markdown((OUTPUT_DIR / 'summary.md').read_text(encoding='utf-8')))